# 05 - Final Load Preparation
## Retail Analytics | SectionB Group 3

This notebook prepares the cleaned dataset for machine learning:
- Drop irrelevant identifier columns
- Label-encode all categorical variables
- Standard-scale numerical features
- Validate zero nulls
- Perform 80/20 train/test split
- Save final CSVs to `data/processed/`


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# ── Resolve project root (works locally and on Colab)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / 'cleaned_dataset.csv'

df = pd.read_csv(INPUT_PATH)
print('Loaded:', df.shape)
print(df.dtypes)


## 1. Drop Irrelevant Columns


In [ ]:
# Drop identifier columns — not useful for modelling
drop_cols = ['Transaction_ID', 'Customer_ID']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print('Shape after dropping identifiers:', df.shape)


## 2. Label Encode Categorical Variables


In [ ]:
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object', 'bool']).columns.tolist()

# Convert bool columns first
bool_cols = df.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)

# Label encode all remaining object columns
obj_cols = df.select_dtypes(include='object').columns.tolist()
label_mappings = {}
for col in obj_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    label_mappings[col] = dict(zip(le.classes_, le.transform(le.classes_)))

print('Encoded columns:', obj_cols)
print('Shape:', df.shape)
print(df.dtypes)


## 3. Final Null Check


In [ ]:
null_count = df.isnull().sum().sum()
print(f'Total null values: {null_count}')
assert null_count == 0, f'Dataset still contains {null_count} null values — fix upstream cleaning step!'
print('✓ No null values found. Dataset is ready for modelling.')


## 4. Scale Numerical Features


In [ ]:
# Identify numerical columns to scale (exclude already-encoded categoricals and binary flags)
skip_scaling = ['order_success', 'is_weekend', 'day']
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [c for c in num_cols if c not in skip_scaling]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print('Scaled columns:', scale_cols)
print(df[scale_cols].describe().round(3))


## 5. Train / Test Split (80 / 20)


In [ ]:
TARGET = 'Total_Amount'
feature_cols = [c for c in df.columns if c != TARGET]

X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set:   {X_train.shape[0]:,} rows')
print(f'Test set:       {X_test.shape[0]:,} rows')


## 6. Save Final Datasets


In [ ]:
# Save full prepared dataset
final_path = PROCESSED_DIR / 'final_prepared_dataset.csv'
df.to_csv(final_path, index=False)
print(f'Saved full prepared dataset → {final_path}')

# Save train set
train_df = X_train.copy()
train_df[TARGET] = y_train.values
train_path = PROCESSED_DIR / 'train_set.csv'
train_df.to_csv(train_path, index=False)
print(f'Saved training set          → {train_path}')

# Save test set
test_df = X_test.copy()
test_df[TARGET] = y_test.values
test_path = PROCESSED_DIR / 'test_set.csv'
test_df.to_csv(test_path, index=False)
print(f'Saved test set              → {test_path}')

print('\n✓ All files saved successfully.')
